# Graphstorm PyTorch Lightning Demonstration

In this notebook, we'll demonstrate how to use Graphstorm with [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable).

---

## Setup 

This notebook requires installing graphstorm using pip. Please find [more details on installation of graphstorm](https://graphstorm.readthedocs.io/en/latest/install/env-setup.html#setup-graphstorm-with-pip-packages).

In [1]:
# make sure DGL is installed
# !pip install dgl -f https://data.dgl.ai/wheels/repo.html

In [2]:
# uncomment if Graphstorm is not installed
# !pip install git+https://github.com/awslabs/graphstorm

In [3]:
import yaml
import graphstorm as gs
import graphstorm.lightning as gsl
import pytorch_lightning as pl
from pathlib import Path

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
num_nodes = 1

# Data Preparation

In this notebook we'll create ACM graph dataset following [this guide](https://graphstorm.readthedocs.io/en/latest/tutorials/own-data.html).

In [5]:
examples = Path(gs.__file__).parent.parent.parent / "examples"
acm_raw = "/tmp/acm_raw"
if not Path(acm_raw).exists():
    acm_data = examples / "acm_data.py"
    !python {acm_data} --output-path {acm_raw}

Namespace(download_path='/tmp/ACM.mat', dataset_name='acm', output_type='raw', output_path='/tmp/acm_raw')
Graph(num_nodes={'author': 17431, 'paper': 12499, 'subject': 73},
      num_edges={('author', 'writing', 'paper'): 37055, ('paper', 'cited', 'paper'): 30789, ('paper', 'citing', 'paper'): 30789, ('paper', 'is-about', 'subject'): 12499, ('paper', 'written-by', 'author'): 37055, ('subject', 'has', 'paper'): 12499},
      metagraph=[('author', 'paper', 'writing'), ('paper', 'paper', 'cited'), ('paper', 'paper', 'citing'), ('paper', 'subject', 'is-about'), ('paper', 'author', 'written-by'), ('subject', 'paper', 'has')])

 Number of classes: 14

 Paper node labels: torch.Size([12499])

 ('paper', 'citing', 'paper') edge labels:30789
Saving ACM data to /tmp/acm.dgl ......
/tmp/acm.dgl saved.
Saving ACM node text to /tmp/acm_text.pkl ......
/tmp/acm_text.pkl saved.
author nodes have: Index(['node_id', 'feat'], dtype='object') columns ......
paper nodes have: Index(['node_id', 'label', 'f

In [6]:
acm_gs = "/tmp/acm_gs"
if not Path(acm_gs).exists():
    !python -m graphstorm.gconstruct.construct_graph \
              --conf-file {acm_raw}/config.json \
              --output-dir {acm_gs} \
              --num-parts {num_nodes} \
              --graph-name acm

INFO:root:The graph has 3 node types and 6 edge types.
INFO:root:Node type author has 17431 nodes
INFO:root:Node type paper has 12499 nodes
INFO:root:Node type subject has 73 nodes
INFO:root:Edge type ('author', 'writing', 'paper') has 37055 edges
INFO:root:Edge type ('paper', 'cited', 'paper') has 30789 edges
INFO:root:Edge type ('paper', 'citing', 'paper') has 30789 edges
INFO:root:Edge type ('paper', 'is-about', 'subject') has 12499 edges
INFO:root:Edge type ('paper', 'written-by', 'author') has 37055 edges
INFO:root:Edge type ('subject', 'has', 'paper') has 12499 edges
INFO:root:Node type author has features: ['feat'].
INFO:root:Node type paper has features: ['feat', 'train_mask', 'val_mask', 'test_mask', 'label'].
INFO:root:Train/val/test on paper: 9999, 1249, 1249
INFO:root:Node type subject has features: ['feat'].
INFO:root:Edge type ('paper', 'citing', 'paper') has features: ['train_mask', 'val_mask', 'test_mask'].
INFO:root:Train/val/test on ('paper', 'citing', 'paper'): 24631

In [7]:
# Works in Jupyter, Colab and Kaggle!
trainer = pl.Trainer(accelerator="auto", devices="auto", max_epochs=2)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [8]:
config = yaml.safe_load(f"""
  gsf:
    basic:
      graph_name: acm
      part_config: /tmp/acm_nc_{num_nodes}p/acm.json
      model_encoder_type: rgcn
    gnn:
      fanout: "15,10"
      num_layers: 2
      hidden_size: 128
      use_mini_batch_infer: false
    input:
      restore_model_path: null
    output:
      save_model_path: null
      save_embed_path: null
    hyperparam:
      dropout: 0.5
      lr: 0.001
      num_epochs: 10
      batch_size: 1024
      wd_l2norm: 0
    rgcn:
      num_bases: -1
      use_self_loop: true
      sparse_optimizer_lr: 1e-2
      use_node_embeddings: false
    node_classification:
      node_feat_name:
        - paper:feat
        - author:feat
        - subject:feat
      target_ntype: paper
      label_field: label
      multilabel: false
      num_classes: 40
      eval_metric:
        - accuracy
""")

In [9]:
datamodule = gsl.datamodule.GSgnnNodeTrainDataModule(trainer=trainer, config=config, graph_data_uri=acm_gs)

In [10]:
model = gsl.module.GSgnnNodeModel(datamodule=datamodule, config=config)

In [11]:
trainer.fit(model=model, datamodule=datamodule)

/opt/conda/lib/python3.10/site-packages/dgl/distributed/dist_context.py:248: DGLWarning: net_type is deprecated and will be removed in future release.
  dgl_warning(
INFO:root:Start to load partition from /tmp/acm_nc_1p/part0/graph.dgl which is 5055161 bytes. It may take non-trivial time for large partition.
INFO:root:Finished loading partition from /tmp/acm_nc_1p/part0/graph.dgl.
INFO:root:Finished loading node data.
INFO:root:Finished loading edge data.
INFO:root:part 0, train: 9999, val: 1249, test: 1249

  | Name  | Type           | Params
-----------------------------------------
0 | model | GSgnnNodeModel | 333 K 
-----------------------------------------
333 K     Trainable params
0         Non-trainable params
333 K     Total params
1.332     Total estimated model params size (MB)


Initialize the distributed services with graphbolt: False
Sanity Checking: |                                                                                                                              | 0/? [00:00<?, ?it/s]

INFO:root:[Rank 0] dist_inference: finishes 0 iterations.
INFO:root:[Rank 0] dist_inference: finishes 0 iterations.


/opt/conda/lib/python3.10/site-packages/pytorch_lightning/loops/fit_loop.py:298: The number of training batches (10) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:02<00:00,  3.78it/s, v_num=13, train_loss=2.590]
Validation: |                                                                                                                                   | 0/? [00:00<?, ?it/s]

...


Epoch 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:03<00:00,  2.76it/s, v_num=13, train_loss=1.820]
Validation: |                                                                                                                                   | 0/? [00:00<?, ?it/s]

INFO:root:[Rank 0] dist_inference: finishes 0 iterations.
INFO:root:[Rank 0] dist_inference: finishes 0 iterations.



Epoch 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:05<00:00,  1.82it/s, v_num=13, train_loss=1.820]

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:05<00:00,  1.82it/s, v_num=13, train_loss=1.820]
